In [24]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from IPython.display import display
warnings.filterwarnings("ignore")
# Surprise imports for collaborative filtering
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, GridSearchCV
from surprise import accuracy
# For evaluation metrics
from collections import defaultdict

In [26]:
# Load data from local paths (update as needed)
books = pd.read_csv('../data/original/Books.csv')
users = pd.read_csv('../data/original/Users.csv')
ratings = pd.read_csv('../data/original/Ratings.csv')

In [27]:
# Quick data overview
print('Books:', books.shape)
print('Ratings:', ratings.shape)
print('Users:', users.shape)
print('Books missing values:', books.isnull().sum().sum())
print('Ratings missing values:', ratings.isnull().sum().sum())
print('Users missing values:', users.isnull().sum().sum())

Books: (271360, 8)
Ratings: (1048575, 3)
Users: (278858, 3)
Books missing values: 7
Ratings missing values: 0
Users missing values: 110762


In [29]:
# Analyze missing values in Users and Books
print('Books missing by column:')
print(books.isnull().sum())
print('\nUsers missing by column:')
print(users.isnull().sum())

# Drop rows with missing critical info in Books (e.g., title, author)
books_cleaned = books.dropna(subset=['Book-Title', 'Book-Author'])

# For Users, drop rows missing all info, or impute if only age/location is missing
users_cleaned = users.dropna(how='all')
users_cleaned['Age'].fillna(users_cleaned['Age'].median(), inplace=True)
users_cleaned['Location'].fillna('Unknown', inplace=True)

print('Books after cleaning:', books_cleaned.shape)
print('Users after cleaning:', users_cleaned.shape)

Books missing by column:
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

Users missing by column:
User-ID          0
Location         0
Age         110762
dtype: int64
Books after cleaning: (271358, 8)
Users after cleaning: (278858, 3)


In [30]:
# Merge ratings with book titles
ratings_with_book_titles = ratings.merge(books, on='ISBN', how='left')

In [31]:
# Preview merged ratings
ratings_with_book_titles.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
1,276726,155061224,5,Rites of Passage,Judith Rae,2001,Heinle,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...
2,276727,446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...
4,276729,521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...


In [32]:
# Drop unnecessary columns
ratings_with_book_titles.drop(columns=["ISBN","Image-URL-S","Image-URL-M"], axis=1, inplace=True)

In [33]:
# Check cleaned ratings
ratings_with_book_titles.head()

,User-ID,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-L
0,276725,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...
1,276726,5,Rites of Passage,Judith Rae,2001,Heinle,http://images.amazon.com/images/P/0155061224.0...
2,276727,0,The Notebook,Nicholas Sparks,1996,Warner Books,http://images.amazon.com/images/P/0446520802.0...
3,276729,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,http://images.amazon.com/images/P/052165615X.0...
4,276729,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,http://images.amazon.com/images/P/0521795028.0...


### Collaborative Filtering

In [34]:
# Filter active users and popular books for better recommendations
rate_threshold = 180  # users with more than 180 ratings
num_ratings_per_user = ratings_with_book_titles.groupby('User-ID')['Book-Rating'].count()
user_ids = num_ratings_per_user[num_ratings_per_user > rate_threshold].index
user_ratings = ratings_with_book_titles[ratings_with_book_titles['User-ID'].isin(user_ids)]
min_rate_count_thresh = 50  # books with at least 50 ratings
rating_counts = user_ratings.groupby('Book-Title').count()['Book-Rating']
popular_books = rating_counts[rating_counts >= min_rate_count_thresh].index
final_ratings = user_ratings[user_ratings['Book-Title'].isin(popular_books)]

In [35]:
# Filter ratings from user_ids
user_ratings =ratings_with_book_titles[ratings_with_book_titles['User-ID'].isin(user_ids)]

In [36]:
min_rate_count_thresh=50
rating_counts= user_ratings.groupby('Book-Title').count()['Book-Rating']
popular_books = rating_counts[rating_counts >=min_rate_count_thresh].index

In [37]:
final_ratings = user_ratings[user_ratings['Book-Title'].isin(popular_books)]

In [38]:
# Create user-item pivot table (item-based filtering)
pivot_table = final_ratings.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating')
pivot_table.fillna(0, inplace=True)
pivot_table.head()

User-ID,254,2033,2276,2766,2977,3363,3757,4017,4385,6242,...,249862,249894,250184,250405,250764,277427,277478,277639,278188,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
pivot_table.fillna(0,inplace=True)
pivot_table

User-ID,254,2033,2276,2766,2977,3363,4017,4385,6251,6323,...,249862,249894,250184,250405,250764,277427,277478,277639,278188,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Bend in the Road,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### cosine_similarity 
matrix as input, each rows represent a data point and  columns represent a feature. 
So in my data, rows represent users,  columns represent book titles. 
Function calculates the cosine similarity between every pair of users in the matrix, measuring angle between two vectors;
a 1 score shows perfect similarity, 
and 0 shows perfect dissimilarity. 
Output is square matrix where each element (i, j) represents the cosine similarity score between user i and user j.

Use this matrix to recommend items to users based on their similarity to other users who have rated those books highly.
For example, find the user with the highest cosine similarity to a particular user and recommend the items that the similar user rated highly.

In [39]:
# Calculate cosine similarity between books (item-based)
from sklearn.metrics.pairwise import cosine_similarity
similarity_score = cosine_similarity(pivot_table)

In [40]:
similarity_score = cosine_similarity(pivot_table)

In [41]:
# Improved recommendation function with error handling and tabulated output
from tabulate import tabulate

def recommend(book_name, top_n=5):
    if book_name not in pivot_table.index:
        print(f"Book '{book_name}' not found in data.")
        return []
    index = np.where(pivot_table.index == book_name)[0][0]
    similar_books = sorted(list(enumerate(similarity_score[index])), key=lambda x: x[1], reverse=True)[1:top_n+1]
    data = []
    for i in similar_books:
        temp_df = books[books['Book-Title'] == pivot_table.index[i[0]]]
        title = temp_df['Book-Title'].values[0] if not temp_df.empty else pivot_table.index[i[0]]
        author = temp_df['Book-Author'].values[0] if not temp_df.empty else 'Unknown'
        similarity = i[1]
        data.append([title, author, similarity])
    print(tabulate(data, headers=["Book Title", "Author", "Similarity Score"], tablefmt="grid"))
    return data

In [42]:
# Example: Recommend similar books to 'The Alienist'
recommend("The Alienist", top_n=5)

+-----------------------------+---------------------------+--------------------+
| Book Title                  | Author                    |   Similarity Score |
+=============================+===========================+====================+
| The Poisonwood Bible        | Barbara Kingsolver        |           0.292772 |
+-----------------------------+---------------------------+--------------------+
| The Angel of Darkness       | Caleb Carr                |           0.286836 |
+-----------------------------+---------------------------+--------------------+
| The Cradle Will Fall        | Mary Higgins Clark        |           0.277298 |
+-----------------------------+---------------------------+--------------------+
| The Shipping News : A Novel | Annie Proulx              |           0.25832  |
+-----------------------------+---------------------------+--------------------+
| Postmortem                  | Patricia Daniels Cornwell |           0.233253 |
+---------------------------

[['The Poisonwood Bible', 'Barbara Kingsolver', 0.29277178258202374],
 ['The Angel of Darkness', 'Caleb Carr', 0.28683598188505244],
 ['The Cradle Will Fall', 'Mary Higgins Clark', 0.2772977360571488],
 ['The Shipping News : A Novel', 'Annie Proulx', 0.25832041748398893],
 ['Postmortem', 'Patricia Daniels Cornwell', 0.23325276448749566]]

## Evaluation Martix

Singular Value Decomposition is used for collaborative filtering based on matrix factorization. It decomposes the user-item rating matrix into two smaller matrices:

User latent factors: These represent "underlying preferences" or hidden characteristics of users.
Item latent factors: These represent "intrinsic features" or characteristics of items. When multiplied together, these two matrices approximate the original rating matrix.

In [43]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# Define the rating scale
reader = Reader(rating_scale=(0, 10))

# Load the data into Surprise's dataset format
data = Dataset.load_from_df(final_ratings[['User-ID', 'Book-Title', 'Book-Rating']], reader)

# Split the dataset into training and testing sets
train_set, test_set = train_test_split(data, test_size=0.20, random_state=42)

# Define the SVD algorithm
model = SVD()

# Train the algorithm on the training set
model.fit(train_set)

# Make predictions on the test set
predictions = model.test(test_set)

# Evaluate the model
accuracy.rmse(predictions)

RMSE: 3.5038


3.503811328819776

In [44]:
# Hyperparameter tuning for SVD using GridSearchCV from Surprise
from surprise.model_selection import GridSearchCV
param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.05]
}
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)
print(f"Best RMSE: {gs.best_score['rmse']}")
print(f"Best MAE: {gs.best_score['mae']}")
print(f"Best Params: {gs.best_params['rmse']}")
best_model = gs.best_estimator['rmse']
train_set, test_set = train_test_split(data, test_size=0.20, random_state=42)
best_model.fit(train_set)
predictions = best_model.test(test_set)
from surprise import accuracy
print("RMSE:", accuracy.rmse(predictions))
print("MAE:", accuracy.mae(predictions))

Best RMSE: 3.295731636270752
Best MAE: 2.5289125990122865
Best Params: {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.002, 'reg_all': 0.05}
RMSE: 3.3069
RMSE: 3.3069395856225547
MAE:  2.5618
MAE: 2.561830127890366


In [46]:
# Precision@k and Recall@k evaluation for recommendations
from collections import defaultdict
def precision_recall_at_k(predictions, k=10, threshold=7):
    '''Return precision and recall at k metrics for each user'''
    # Map the predictions to each user.
    user_est_true = defaultdict(list)
    for pred in predictions:
        user_est_true[pred.uid].append((pred.est, pred.r_ui))
    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():
        # Sort user ratings by estimated value
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        # Top-k predictions
        top_k = user_ratings[:k]
        # Count relevant items in top-k
        n_rel = sum((true_rating >= threshold) for (_, true_rating) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in top_k)
        n_rel_and_rec_k = sum(((true_rating >= threshold) and (est >= threshold)) for (est, true_rating) in top_k)
        # Precision@k: Proportion of recommended items that are relevant
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        # Recall@k: Proportion of relevant items that are recommended
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0
    # Average precision and recall over all users
    avg_precision = sum(precisions.values()) / len(precisions)
    avg_recall = sum(recalls.values()) / len(recalls)
    print(f'Precision@{k}: {avg_precision:.4f}')
    print(f'Recall@{k}: {avg_recall:.4f}')
    return avg_precision, avg_recall

# Evaluate precision@k and recall@k for best_model
precision_recall_at_k(predictions, k=10, threshold=7)

Precision@10: 0.0181
Recall@10: 0.0058


(0.018145161290322582, 0.0057760375848646805)

In [47]:
def recommend_books(user_id, n=10):
    # List all unique book titles
    all_books = final_ratings['Book-Title'].unique()

    # Remove books already rated by the user
    rated_books = final_ratings[final_ratings['User-ID'] == user_id]['Book-Title'].values
    books_to_predict = [book for book in all_books if book not in rated_books]

    # Predict ratings for remaining books
    predictions = []
    for book in books_to_predict:
        pred = model.predict(user_id, book)
        predictions.append((book, pred.est))

    # Sort predictions by estimated rating
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get top N recommendations
    top_n = predictions[:n]

    return top_n


In [48]:

user_id = 271705
recommended_books = recommend_books(user_id)
print(f"Top {len(recommended_books)} recommended books for user {user_id}:")
for i, (title, similarity_score) in enumerate(recommended_books, start=1):
    print(f"{i}. {title} (Similarity Score: {similarity_score})")


Top 10 recommended books for user 271705:
1. Harry Potter and the Order of the Phoenix (Book 5) (Similarity Score: 4.319856942187592)
2. 84 Charing Cross Road (Similarity Score: 4.28165553002957)
3. Harry Potter and the Sorcerer's Stone (Book 1) (Similarity Score: 4.19542125266522)
4. Stupid White Men ...and Other Sorry Excuses for the State of the Nation! (Similarity Score: 4.134495584741711)
5. The Little Prince (Similarity Score: 4.119957410866552)
6. Harry Potter and the Goblet of Fire (Book 4) (Similarity Score: 4.0061155185467685)
7. Harry Potter and the Prisoner of Azkaban (Book 3) (Similarity Score: 3.861838101711782)
8. The Secret Garden (Similarity Score: 3.833785317234383)
9. Anne Frank: The Diary of a Young Girl (Similarity Score: 3.7140871744347326)
10. The Two Towers (The Lord of the Rings, Part 2) (Similarity Score: 3.61806480225965)
